In [ ]:
# =====================================================================
# PHYSICAL AI SMART FACTORY DEMO #2
# PREDICTIVE MAINTENANCE WITH GEMINI
#
# GOOGLE COLAB - SINGLE CELL
#
# Concept:
#
# Industrial Motor
#       ↓
# Sensors
# Temperature / Vibration / Current / RPM
#       ↓
# Gemini LLM
#       ↓
# NORMAL / MONITOR / SLOW_MACHINE / STOP_FOR_MAINTENANCE
#       ↓
# Safety Controller
#       ↓
# Physical Motor Controller
#       ↓
# Motor speed changes / machine stops
#       ↓
# Feedback
#
# EDUCATIONAL SIMULATION ONLY
# =====================================================================


# =====================================================================
# 1. INSTALL
# =====================================================================

!pip install -q google-genai opencv-python


# =====================================================================
# 2. IMPORTS
# =====================================================================

import cv2
import numpy as np
import random
import math
import json
import os

from google import genai
from google.colab import userdata

from IPython.display import HTML, display
from base64 import b64encode


# =====================================================================
# 3. GEMINI SETUP
# =====================================================================

USE_GEMINI = False
client = None

# =====================================================================
# STEP 2
# GEMINI SETUP
#
# In Colab:
#
# Left sidebar
#      ↓
# Secrets
#      ↓
# Add:
#
# GEMINI_API_KEY
#
# =====================================================================

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [ ]:
# =====================================================================
# 4. VIDEO SETTINGS
# =====================================================================

WIDTH = 1280
HEIGHT = 720

FPS = 25

DURATION = 25

TOTAL_FRAMES = FPS * DURATION


raw_video = (
    "predictive_maintenance_raw.mp4"
)

final_video = (
    "predictive_maintenance_demo.mp4"
)


writer = cv2.VideoWriter(

    raw_video,

    cv2.VideoWriter_fourcc(
        *"mp4v"
    ),

    FPS,

    (
        WIDTH,
        HEIGHT
    )

)


# =====================================================================
# 5. MACHINE STATE
# =====================================================================

motor_rpm = 1450.0

target_rpm = 1450.0


temperature = 48.0

vibration = 1.4

current = 13.5


bearing_health = 100.0


motor_angle = 0.0


gemini_decision = "NORMAL"

gemini_reason = (
    "Machine operating normally"
)

final_action = "NORMAL"

safety_status = "APPROVED"


maintenance_alert = False


# =====================================================================
# 6. SENSOR HISTORY
# =====================================================================

temperature_history = []

vibration_history = []

rpm_history = []


# =====================================================================
# 7. MACHINE SENSOR SIMULATION
# =====================================================================

def update_sensors(frame_number):

    global temperature
    global vibration
    global current
    global bearing_health


    # -------------------------------------------------------------
    # Normal gradual bearing degradation
    # -------------------------------------------------------------

    bearing_health -= 0.018


    # -------------------------------------------------------------
    # As bearing health drops,
    # vibration gradually increases
    # -------------------------------------------------------------

    degradation = (
        100
        -
        bearing_health
    )


    vibration_base = (
        1.4
        +
        degradation
        *
        0.035
    )


    vibration = (
        vibration_base
        +
        random.uniform(
            -0.08,
            0.08
        )
    )


    # -------------------------------------------------------------
    # Temperature rises with degradation
    # -------------------------------------------------------------

    temperature_base = (
        48
        +
        degradation
        *
        0.20
    )


    temperature = (
        temperature_base
        +
        random.uniform(
            -0.25,
            0.25
        )
    )


    # -------------------------------------------------------------
    # Motor current depends partly on mechanical stress
    # -------------------------------------------------------------

    current = (

        13.5

        +

        degradation
        *
        0.03

        +

        random.uniform(
            -0.15,
            0.15
        )

    )


    # =============================================================
    # Inject a visible fault later in simulation
    # =============================================================

    if frame_number > FPS * 12:

        vibration += (
            frame_number
            -
            FPS*12
        ) * 0.003


    if frame_number > FPS * 16:

        temperature += (
            frame_number
            -
            FPS*16
        ) * 0.006


# =====================================================================
# 8. FALLBACK AI DECISION
# =====================================================================

def fallback_decision():

    if (
        temperature >= 74
        or
        vibration >= 6.0
    ):

        return (

            "STOP_FOR_MAINTENANCE",

            "Critical temperature or vibration"

        )


    elif (
        temperature >= 64
        or
        vibration >= 4.5
    ):

        return (

            "SLOW_MACHINE",

            "Machine degradation detected"

        )


    elif (
        temperature >= 57
        or
        vibration >= 3.0
    ):

        return (

            "MONITOR",

            "Early abnormal condition detected"

        )


    else:

        return (

            "NORMAL",

            "Sensor readings normal"

        )


# =====================================================================
# 9. GEMINI PREDICTIVE MAINTENANCE DECISION
# =====================================================================

def ask_gemini():

    if not USE_GEMINI:

        return fallback_decision()


    sensor_state = {

        "motor_rpm":
            round(
                motor_rpm,
                1
            ),

        "temperature_c":
            round(
                temperature,
                1
            ),

        "vibration_mm_s":
            round(
                vibration,
                2
            ),

        "motor_current_a":
            round(
                current,
                2
            ),

        "bearing_health_percent":
            round(
                bearing_health,
                1
            )

    }


    prompt = f"""

You are the HIGH-LEVEL predictive maintenance
reasoning module of an EDUCATIONAL Physical AI
smart factory.

Industrial motor sensor state:

{json.dumps(sensor_state, indent=2)}

Choose exactly ONE action:

NORMAL
MONITOR
SLOW_MACHINE
STOP_FOR_MAINTENANCE

Guidance:

NORMAL:
Sensor values look healthy.

MONITOR:
Early degradation exists but machine can continue.

SLOW_MACHINE:
Significant abnormal temperature or vibration exists.

STOP_FOR_MAINTENANCE:
Critical condition or high probability of damage.

Safety and equipment protection are more important
than production speed.

Return exactly:

ACTION|short reason

Example:

SLOW_MACHINE|Vibration is increasing and bearing health is declining

"""


    try:

        response = client.models.generate_content(

            model="gemini-2.5-flash",

            contents=prompt

        )


        text = (
            response.text
            or ""
        ).strip()


        if "|" not in text:

            return fallback_decision()


        action, reason = text.split(
            "|",
            1
        )


        action = action.strip().upper()

        reason = reason.strip()


        valid = [

            "NORMAL",

            "MONITOR",

            "SLOW_MACHINE",

            "STOP_FOR_MAINTENANCE"

        ]


        if action not in valid:

            return fallback_decision()


        return action, reason


    except Exception as e:

        print(
            "Gemini call failed:",
            e
        )

        return fallback_decision()


# =====================================================================
# 10. SAFETY CONTROLLER
# =====================================================================

def safety_controller(ai_action):


    # -------------------------------------------------------------
    # Hard safety limits
    # -------------------------------------------------------------

    if temperature >= 78:

        return (

            "STOP_FOR_MAINTENANCE",

            "OVERRIDE: CRITICAL TEMPERATURE"

        )


    if vibration >= 6.5:

        return (

            "STOP_FOR_MAINTENANCE",

            "OVERRIDE: CRITICAL VIBRATION"

        )


    # -------------------------------------------------------------
    # Strong warning
    # -------------------------------------------------------------

    if (

        vibration >= 5

        and

        ai_action in [
            "NORMAL",
            "MONITOR"
        ]

    ):

        return (

            "SLOW_MACHINE",

            "OVERRIDE: VIBRATION TOO HIGH"

        )


    return (

        ai_action,

        "APPROVED"

    )


# =====================================================================
# 11. PHYSICAL MOTOR CONTROLLER
# =====================================================================

def motor_controller(action):

    global target_rpm
    global maintenance_alert


    maintenance_alert = False


    if action == "NORMAL":

        target_rpm = 1450


    elif action == "MONITOR":

        target_rpm = 1350


    elif action == "SLOW_MACHINE":

        target_rpm = 800


    elif action == "STOP_FOR_MAINTENANCE":

        target_rpm = 0

        maintenance_alert = True


# =====================================================================
# 12. MOTOR PHYSICS
# =====================================================================

def update_motor():

    global motor_rpm
    global motor_angle


    if motor_rpm < target_rpm:

        motor_rpm += 18


    elif motor_rpm > target_rpm:

        motor_rpm -= 28


    motor_rpm = max(
        0,
        motor_rpm
    )


    # -------------------------------------------------------------
    # Rotation speed visually reflects RPM
    # -------------------------------------------------------------

    motor_angle += (
        motor_rpm
        /
        1450
        *
        18
    )


# =====================================================================
# 13. DRAW FACTORY ENVIRONMENT
# =====================================================================

def draw_factory(frame):


    frame[:] = (
        235,
        235,
        235
    )


    # floor

    cv2.rectangle(

        frame,

        (0,520),

        (WIDTH,HEIGHT),

        (210,210,210),

        -1

    )


    # title

    cv2.putText(

        frame,

        "PHYSICAL AI - PREDICTIVE MAINTENANCE",

        (65,55),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.95,

        (20,20,20),

        3,

        cv2.LINE_AA

    )


# =====================================================================
# 14. DRAW INDUSTRIAL MOTOR
# =====================================================================

def draw_motor(frame):


    motor_x = 610

    motor_y = 365


    # =============================================================
    # MACHINE BASE
    # =============================================================

    cv2.rectangle(

        frame,

        (430,470),

        (790,520),

        (65,65,65),

        -1

    )


    # =============================================================
    # MOTOR BODY
    # =============================================================

    cv2.rectangle(

        frame,

        (470,300),

        (720,470),

        (80,120,150),

        -1

    )


    # motor cooling ribs

    for x in range(
        490,
        700,
        25
    ):

        cv2.line(

            frame,

            (x,310),

            (x,460),

            (45,80,105),

            6

        )


    # =============================================================
    # SHAFT
    # =============================================================

    cv2.rectangle(

        frame,

        (720,355),

        (850,385),

        (90,90,90),

        -1

    )


    # =============================================================
    # ROTATING FAN / SHAFT DISC
    # =============================================================

    center = (
        850,
        370
    )


    radius = 55


    cv2.circle(

        frame,

        center,

        radius,

        (55,55,55),

        5

    )


    # rotating blades

    for angle_offset in [
        0,
        90,
        180,
        270
    ]:


        angle = math.radians(
            motor_angle
            +
            angle_offset
        )


        x2 = int(
            center[0]
            +
            radius*0.8
            *
            math.cos(angle)
        )


        y2 = int(
            center[1]
            +
            radius*0.8
            *
            math.sin(angle)
        )


        cv2.line(

            frame,

            center,

            (
                x2,
                y2
            ),

            (40,40,40),

            9

        )


    # center hub

    cv2.circle(

        frame,

        center,

        13,

        (30,30,30),

        -1

    )


    # =============================================================
    # MOTOR LABEL
    # =============================================================

    cv2.putText(

        frame,

        "INDUSTRIAL MOTOR M-01",

        (485,280),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (20,20,20),

        2

    )


# =====================================================================
# 15. DRAW SENSOR ICONS
# =====================================================================

def draw_sensors(frame):


    # =============================================================
    # TEMPERATURE SENSOR
    # =============================================================

    cv2.circle(

        frame,

        (500,240),

        19,

        (0,0,220),

        -1

    )


    cv2.putText(

        frame,

        "TEMP SENSOR",

        (435,205),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.47,

        (20,20,20),

        1

    )


    # =============================================================
    # VIBRATION SENSOR
    # =============================================================

    cv2.rectangle(

        frame,

        (590,220),

        (630,255),

        (30,170,220),

        -1

    )


    cv2.putText(

        frame,

        "VIBRATION",

        (565,205),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.47,

        (20,20,20),

        1

    )


    # =============================================================
    # CURRENT SENSOR
    # =============================================================

    cv2.circle(

        frame,

        (700,240),

        19,

        (200,90,30),

        -1

    )


    cv2.putText(

        frame,

        "CURRENT",

        (670,205),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.47,

        (20,20,20),

        1

    )


# =====================================================================
# 16. DRAW LEFT SENSOR PANEL
# =====================================================================

def draw_sensor_panel(frame):


    # -------------------------------------------------------------
    # panel
    # -------------------------------------------------------------

    cv2.rectangle(

        frame,

        (25,90),

        (385,385),

        (25,25,25),

        -1

    )


    cv2.rectangle(

        frame,

        (25,90),

        (385,385),

        (75,75,75),

        3

    )


    cv2.putText(

        frame,

        "MACHINE SENSOR DATA",

        (45,125),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.65,

        (80,255,80),

        2

    )


    # -------------------------------------------------------------
    # Health color
    # -------------------------------------------------------------

    if bearing_health > 70:

        health_color = (
            0,
            255,
            0
        )

    elif bearing_health > 40:

        health_color = (
            0,
            200,
            255
        )

    else:

        health_color = (
            0,
            0,
            255
        )


    rows = [

        f"RPM            : {motor_rpm:.0f}",

        f"Temperature    : {temperature:.1f} C",

        f"Vibration      : {vibration:.2f} mm/s",

        f"Motor Current  : {current:.2f} A",

        f"Bearing Health : {bearing_health:.1f} %"

    ]


    y = 175


    for row in rows:

        cv2.putText(

            frame,

            row,

            (45,y),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.52,

            (245,245,245),

            1

        )

        y += 38


    # -------------------------------------------------------------
    # Bearing health bar
    # -------------------------------------------------------------

    cv2.rectangle(

        frame,

        (45,340),

        (345,360),

        (80,80,80),

        -1

    )


    health_width = int(

        300
        *
        max(
            bearing_health,
            0
        )
        /
        100

    )


    cv2.rectangle(

        frame,

        (45,340),

        (
            45+health_width,
            360
        ),

        health_color,

        -1

    )


# =====================================================================
# 17. DRAW GEMINI PANEL
# =====================================================================

def draw_gemini_panel(frame):


    cv2.rectangle(

        frame,

        (885,90),

        (1250,365),

        (248,248,248),

        -1

    )


    cv2.rectangle(

        frame,

        (885,90),

        (1250,365),

        (55,55,55),

        3

    )


    cv2.putText(

        frame,

        "GEMINI MAINTENANCE AI",

        (905,125),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.58,

        (180,60,20),

        2

    )


    rows = [

        f"Gemini : {gemini_decision}",

        f"Final  : {final_action}",

        f"Safety : {safety_status}",

        "",

        "Reason:",

        gemini_reason[:38]

    ]


    y = 170


    for row in rows:

        cv2.putText(

            frame,

            row,

            (905,y),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.46,

            (25,25,25),

            1

        )

        y += 34


# =====================================================================
# 18. DRAW MAINTENANCE INDICATOR
# =====================================================================

def draw_machine_status(frame):


    if final_action == "NORMAL":

        status = "MACHINE RUNNING"

        color = (
            0,
            170,
            0
        )


    elif final_action == "MONITOR":

        status = "AI MONITORING"

        color = (
            0,
            180,
            220
        )


    elif final_action == "SLOW_MACHINE":

        status = "MACHINE SPEED REDUCED"

        color = (
            0,
            120,
            255
        )


    else:

        status = "MACHINE STOPPED - MAINTENANCE REQUIRED"

        color = (
            0,
            0,
            255
        )


    cv2.rectangle(

        frame,

        (400,565),

        (870,625),

        (245,245,245),

        -1

    )


    cv2.rectangle(

        frame,

        (400,565),

        (870,625),

        color,

        3

    )


    cv2.putText(

        frame,

        status,

        (420,605),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        color,

        2

    )


# =====================================================================
# 19. DRAW SIMPLE SENSOR TREND GRAPH
# =====================================================================

def draw_trend_graph(frame):


    graph_x1 = 60

    graph_y1 = 440

    graph_x2 = 350

    graph_y2 = 600


    cv2.rectangle(

        frame,

        (
            graph_x1,
            graph_y1
        ),

        (
            graph_x2,
            graph_y2
        ),

        (250,250,250),

        -1

    )


    cv2.rectangle(

        frame,

        (
            graph_x1,
            graph_y1
        ),

        (
            graph_x2,
            graph_y2
        ),

        (100,100,100),

        2

    )


    cv2.putText(

        frame,

        "VIBRATION TREND",

        (
            graph_x1+45,
            graph_y1+25
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.46,

        (20,20,20),

        1

    )


    values = vibration_history[-60:]


    if len(values) > 1:


        points = []


        for i,value in enumerate(values):


            px = int(

                graph_x1
                +
                10
                +
                i
                *
                (
                    graph_x2
                    -
                    graph_x1
                    -
                    20
                )
                /
                max(
                    len(values)-1,
                    1
                )

            )


            py = int(

                graph_y2
                -
                10
                -
                min(
                    value/8,
                    1
                )
                *
                (
                    graph_y2
                    -
                    graph_y1
                    -
                    50
                )

            )


            points.append(
                (
                    px,
                    py
                )
            )


        for i in range(
            len(points)-1
        ):

            cv2.line(

                frame,

                points[i],

                points[i+1],

                (0,100,220),

                2

            )


# =====================================================================
# 20. MAIN SIMULATION
# =====================================================================

DECISION_INTERVAL = FPS * 3


for frame_number in range(
    TOTAL_FRAMES
):


    # =============================================================
    # SENSOR UPDATE
    # =============================================================

    update_sensors(
        frame_number
    )


    # =============================================================
    # GEMINI DECISION EVERY 3 SECONDS
    # =============================================================

    if (

        frame_number == 0

        or

        frame_number
        %
        DECISION_INTERVAL
        ==
        0

    ):


        gemini_decision, gemini_reason = (
            ask_gemini()
        )


        final_action, safety_status = (
            safety_controller(
                gemini_decision
            )
        )


        motor_controller(
            final_action
        )


        print()

        print(
            f"Time "
            f"{frame_number/FPS:.1f}s"
        )

        print(
            "Sensors:",
            f"T={temperature:.1f}C,",
            f"V={vibration:.2f},",
            f"Health={bearing_health:.1f}%"
        )

        print(
            "Gemini:",
            gemini_decision
        )

        print(
            "Reason:",
            gemini_reason
        )

        print(
            "Safety:",
            safety_status
        )

        print(
            "Final action:",
            final_action
        )


    # =============================================================
    # PHYSICAL MACHINE CONTROL
    # =============================================================

    motor_controller(
        final_action
    )


    update_motor()


    # =============================================================
    # HISTORY
    # =============================================================

    temperature_history.append(
        temperature
    )

    vibration_history.append(
        vibration
    )

    rpm_history.append(
        motor_rpm
    )


    # =============================================================
    # CREATE FRAME
    # =============================================================

    frame = np.zeros(

        (
            HEIGHT,
            WIDTH,
            3
        ),

        dtype=np.uint8

    )


    draw_factory(
        frame
    )


    draw_motor(
        frame
    )


    draw_sensors(
        frame
    )


    draw_sensor_panel(
        frame
    )


    draw_gemini_panel(
        frame
    )


    draw_machine_status(
        frame
    )


    draw_trend_graph(
        frame
    )


    # =============================================================
    # PHYSICAL AI FLOW
    # =============================================================

    cv2.putText(

        frame,

        "Sense -> Analyze -> Gemini Reason -> Safety -> Control Motor -> Feedback",

        (310,690),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (20,20,20),

        2

    )


    writer.write(
        frame
    )


# =====================================================================
# 21. CLOSE VIDEO
# =====================================================================

writer.release()


# =====================================================================
# 22. CONVERT TO H264
# =====================================================================

os.system(

    f'ffmpeg -y '
    f'-loglevel error '
    f'-i "{raw_video}" '
    f'-vcodec libx264 '
    f'-pix_fmt yuv420p '
    f'"{final_video}"'

)


# =====================================================================
# 23. DISPLAY VIDEO
# =====================================================================

with open(
    final_video,
    "rb"
) as f:

    video_bytes = f.read()


encoded_video = b64encode(
    video_bytes
).decode()


print()

print(
    "PHYSICAL AI PREDICTIVE MAINTENANCE DEMO"
)


display(

    HTML(

        f'''

        <video
            width="1000"
            controls
            autoplay
            loop>

            <source
                src="data:video/mp4;base64,{encoded_video}"
                type="video/mp4">

        </video>

        '''

    )

)